In [ ]:
pip install langgraph

In [ ]:
pip install langchain


In [ ]:
pip install -U langchain-community transformers

In [ ]:
from langchain.llms import HuggingFacePipeline
from transformers import pipeline
from langgraph.graph import StateGraph, END
from langchain_core.tools import tool
from typing import TypedDict, List

# Simple local LLM
pipe = pipeline("text-generation", model="gpt2")
llm = HuggingFacePipeline(pipeline=pipe)

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"It's sunny in {city}!"

# Define the state
class AgentState(TypedDict):
    messages: List[str]
    model_output: dict
    tool_result: str

# Node functions
def call_llm(state):
    messages = state["messages"]
    # Join the list of messages into a single string
    input_text = "\n".join(messages)
    response = llm.invoke(input_text) # Pass the joined string
    # For this example, we're hardcoding a function call output.
    # In a real scenario, you would parse the LLM's response to determine
    # if a tool call is needed and what the arguments are.
    return {"model_output": {"type": "function_call", "name": "get_weather", "args": {"city": "Tokyo"}}}


def dispatcher(state):
    call = state["model_output"]
    if call["type"] == "function_call":
        result = get_weather(call["args"]["city"])
        # Return a dictionary of updates to the state
        return {"tool_result": result}
    # If no function call is detected, we could potentially route to END or another node
    return state


# Build graph
graph = StateGraph(AgentState)
graph.add_node("llm", call_llm)
graph.add_node("dispatcher", dispatcher)
graph.add_edge("llm", "dispatcher")
# Add a conditional edge from dispatcher to decide next step
graph.add_conditional_edges(
    "dispatcher",
    lambda state: "llm" if state.get("tool_result") is None else END,
    {
        "llm": "llm",
        END: END
    }
)


# Set the entry point
graph.set_entry_point("llm")

In [ ]:
# Compile the graph into an executable runtime
app = graph.compile()

# Define the initial state
initial_state = {
    "messages": ["User: What's the weather in Tokyo?"],
    "model_output": {},
    "tool_result": ""
}

# Invoke the graph (run the agent)
final_state = app.invoke(initial_state)

# Print the outputs
print("LLM Output / Function Call:", final_state["model_output"])
print("Tool Result:", final_state["tool_result"])
